<a href="https://colab.research.google.com/github/nishanthxmahesh/CRNN-CAPTCHA-Solver/blob/main/CRNN_CAPTCHA_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

New Complex

In [ ]:
!pip install captcha

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.6/147.6 kB 7.0 MB/s eta 0:00:00


In [ ]:
from captcha.image import ImageCaptcha
import numpy as np
import matplotlib.pyplot as plt
import random
import string
import os

# --- Configuration ---
NUM_IMAGES_TO_GENERATE = 100000 # We need a large dataset to train the model
OUTPUT_DIR = "captcha_dataset_complex"
CAPTCHA_LENGTH = 5
IMAGE_WIDTH = 160
IMAGE_HEIGHT = 60
# --- CRITICAL: Now includes uppercase letters AND digits ---
CHARACTERS = string.ascii_uppercase + string.digits

# Function to generate random text
def random_string(length=CAPTCHA_LENGTH):
    """Generates a random string from the new CHARACTERS set."""
    return ''.join(random.choice(CHARACTERS) for i in range(length))

# Generate and save CAPTCHA images
def generate_captcha_dataset(num_images, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    else:
        # Clear existing images in the directory
        print(f"Directory {output_dir} found. Deleting old images...")
        for f in os.listdir(output_dir):
            os.remove(os.path.join(output_dir, f))
        print("Old images deleted.")


    image = ImageCaptcha(width=IMAGE_WIDTH, height=IMAGE_HEIGHT)
    print(f"Generating {num_images} CAPTCHA images (with digits) using the 'captcha' library...")
    for i in range(num_images):
        captcha_text = random_string()
        captcha_image = image.generate_image(captcha_text)

        filename = os.path.join(output_dir, f"{captcha_text}.png")

        # Handle rare collisions
        while os.path.exists(filename):
            captcha_text = random_string()
            captcha_image = image.generate_image(captcha_text)
            filename = os.path.join(output_dir, f"{captcha_text}.png")

        captcha_image.save(filename)

        if (i + 1) % 1000 == 0: # Log every 1000 images
            print(f"  ... generated {i + 1} / {num_images}")

    print("Dataset generation complete.")

if __name__ == "__main__":
    generate_captcha_dataset(NUM_IMAGES_TO_GENERATE, OUTPUT_DIR)

Directory captcha_dataset_complex found. Deleting old images...
Old images deleted.
Generating 100000 CAPTCHA images (with digits) using the 'captcha' library...
  ... generated 1000 / 100000
  ... generated 2000 / 100000
  ... generated 3000 / 100000
  ... generated 4000 / 100000
  ... generated 5000 / 100000
  ... generated 6000 / 100000
  ... generated 7000 / 100000
  ... generated 8000 / 100000
  ... generated 9000 / 100000
  ... generated 10000 / 100000
  ... generated 11000 / 100000
  ... generated 12000 / 100000
  ... generated 13000 / 100000
  ... generated 14000 / 100000
  ... generated 15000 / 100000
  ... generated 16000 / 100000
  ... generated 17000 / 100000
  ... generated 18000 / 100000
  ... generated 19000 / 100000
  ... generated 20000 / 100000
  ... generated 21000 / 100000
  ... generated 22000 / 100000
  ... generated 23000 / 100000
  ... generated 24000 / 100000
  ... generated 25000 / 100000
  ... generated 26000 / 100000
  ... generated 27000 / 100000
  ... gene

In [ ]:
import os
import numpy as np
import tensorflow as T
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Dense, Dropout,
    Flatten, BatchNormalization, Reshape, Softmax,
    Bidirectional, LSTM
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import string

# --- Configuration (ADAPTED FOR YOUR NEW GENERATOR) ---
DATASET_DIR = "captcha_dataset_complex"
IMAGE_WIDTH = 160
IMAGE_HEIGHT = 60
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits # <-- Includes digits
NUM_CHARACTERS = len(CHARACTERS) # This is now 36
TEST_SPLIT_SIZE = 0.2
BATCH_SIZE = 64
EPOCHS = 100

# --- 1. Data Loading ---

char_to_int = {char: i for i, char in enumerate(CHARACTERS)}
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

def load_paths_and_labels():
    """Loads file paths and labels."""
    print("Loading file paths and labels...")
    filenames = []
    labels = []

    for filename in os.listdir(DATASET_DIR):
        if filename.endswith(".png"):
            label_text = filename.split('.')[0]

            if len(label_text) != CAPTCHA_LENGTH:
                print(f"Skipping file with incorrect length: {filename}")
                continue

            try:
                label_int = [char_to_int[char] for char in label_text]
                label_ohe = [to_categorical(i, num_classes=NUM_CHARACTERS) for i in label_int]
                label_stacked = np.stack(label_ohe)

                filenames.append(os.path.join(DATASET_DIR, filename))
                labels.append(label_stacked)
            except KeyError:
                print(f"Skipping file with unknown character: {filename}")

    return filenames, np.array(labels) # Return labels as numpy array

@T.function
def tf_load_and_augment_image(path, label):
    """Loads, decodes (as COLOR), and augments the image."""
    image_bytes = T.io.read_file(path)
    image = T.io.decode_png(image_bytes, channels=3) # 3-channel (RGB)
    image = T.image.convert_image_dtype(image, T.float32)
    image = T.reshape(image, [IMAGE_HEIGHT, IMAGE_WIDTH, 3])

    # Data Augmentation
    image = T.image.random_brightness(image, max_delta=0.2)
    image = T.image.random_contrast(image, lower=0.8, upper=1.2)
    image = T.clip_by_value(image, 0.0, 1.0)

    return image, label

def create_dataset(filenames, labels, is_training=True):
    """Creates the tf.data.Dataset pipeline."""
    dataset = T.data.Dataset.from_tensor_slices((filenames, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(filenames))

    dataset = dataset.map(tf_load_and_augment_image, num_parallel_calls=T.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=T.data.AUTOTUNE)

    return dataset


# --- 2. Build the CRNN Model (ADAPTED) ---

def build_crnn_model():
    """
    Builds the CRNN architecture adapted for (60, 160, 3) input
    and 36 classes.
    """
    print("Building adapted CRNN model for (60, 160, 3) input...")

    input_layer = Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3), name="image_input")

    # "Eye" (CNN)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x) # Shape: (30, 80, 32)

    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x) # Shape: (15, 40, 64)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x) # Shape: (7, 20, 128)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x) # Shape: (3, 10, 256)

    # "Bridge" (Reshape)
    cnn_output_shape = x.shape[1:]
    new_shape = (cnn_output_shape[1], cnn_output_shape[0] * cnn_output_shape[2])
    x = Reshape(target_shape=new_shape, name="reshape_to_sequence")(x) # (None, 10, 768)

    # "Reader" (RNN)
    x = Bidirectional(LSTM(128, return_sequences=True), name="bidirectional_lstm")(x)

    # "Hand" (Predictor)
    x = Flatten(name="flatten_smart_sequence")(x)
    x = Dense(512, activation='relu', name="dense_features")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    output_neurons = CAPTCHA_LENGTH * NUM_CHARACTERS # 5 * 36 = 180
    x = Dense(output_neurons, name="unified_output_dense")(x)

    x = Reshape((CAPTCHA_LENGTH, NUM_CHARACTERS), name="output_reshape")(x) # (None, 5, 36)
    output_layer = Softmax(axis=-1, name="output_softmax")(x)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    model.summary()
    return model

# --- 3. Train and Test ---

def main():
    # 1. Load paths and labels
    filenames, labels = load_paths_and_labels()

    # 2. Split into training and testing sets
    X_train_paths, X_test_paths, y_train, y_test = train_test_split(
        filenames, labels, test_size=TEST_SPLIT_SIZE, random_state=42
    )

    print(f"Training data: {len(X_train_paths)} samples")
    print(f"Testing data: {len(X_test_paths)} samples")

    # 3. Create the tf.data.Dataset pipelines
    print("Creating data pipelines...")
    train_dataset = create_dataset(X_train_paths, y_train, is_training=True)
    val_dataset = create_dataset(X_test_paths, y_test, is_training=False)

    # 4. Build the model
    model = build_crnn_model()

    # 5. Define callbacks
    print("Defining callbacks (EarlyStopping, ReduceLROnPlateau)...")

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=0.00001
    )

    # 6. Train the model
    print("Training CRNN model on 'captcha' library data (with numbers)...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr]
    )

    # 7. Evaluate the model
    print("\nEvaluating model on test set...")
    metrics = model.evaluate(val_dataset, verbose=0)

    print("\n--- Evaluation ---")
    print(f"Total Loss: {metrics[0]:.4f}")
    print(f"  Overall Character Accuracy: {metrics[1]*100:.2f}%")

    # 8. Save the model
    model.save("captcha_solver_alnum.keras")
    print("\nModel saved to 'captcha_solver_alnum.keras'")

if __name__ == "__main__":
    main()

Loading file paths and labels...
Training data: 80000 samples
Testing data: 20000 samples
Creating data pipelines...
Building adapted CRNN model for (60, 160, 3) input...


Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 60, 160, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 60, 160, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 60, 160, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_33 (MaxPooling2D) │ (None, 30, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_34 (Conv2D)              │ (None, 30, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 30, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_34 (MaxPooling2D) │ (None, 15, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_35 (Conv2D)              │ (None, 15, 40, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_32          │ (None, 15, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_35 (MaxPooling2D) │ (None, 7, 20, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 7, 20, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_33          │ (None, 7, 20, 256)     │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_36 (MaxPooling2D) │ (None, 3, 10, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_to_sequence (Reshape)   │ (None, 10, 768)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_lstm              │ (None, 10, 256)        │       918,528 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_smart_sequence          │ (None, 2560)           │             0 │
│ (Flatten)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_features (Dense)          │ (None, 512)            │     1,311,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_34          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ unified_output_dense (Dense)    │ (None, 180)            │        92,340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_reshape (Reshape)        │ (None, 5, 36)          │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 2,714,484 (10.35 MB)

 Trainable params: 2,712,500 (10.35 MB)

 Non-trainable params: 1,984 (7.75 KB)

Defining callbacks (EarlyStopping, ReduceLROnPlateau)...
Training CRNN model on 'captcha' library data (with numbers)...
Epoch 1/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 94s 71ms/step - accuracy: 0.2636 - loss: 2.9026 - val_accuracy: 0.8634 - val_loss: 0.4731 - learning_rate: 0.0010
Epoch 2/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 86s 69ms/step - accuracy: 0.8871 - loss: 0.3751 - val_accuracy: 0.8824 - val_loss: 0.3949 - learning_rate: 0.0010
Epoch 3/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 88s 71ms/step - accuracy: 0.9335 - loss: 0.2066 - val_accuracy: 0.9534 - val_loss: 0.1382 - learning_rate: 0.0010
Epoch 4/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 140s 69ms/step - accuracy: 0.9499 - loss: 0.1530 - val_accuracy: 0.9348 - val_loss: 0.2147 - learning_rate: 0.0010
Epoch 5/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 85s 68ms/step - accuracy: 0.9587 - loss: 0.1258 - val_accuracy: 0.9214 - val_loss: 0.3143 - learning_rate: 0.0010
Epoch 6/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 89s 71ms/step - accuracy: 0.9643 - loss: 0.1085 - 

In [ ]:
#Predict
import os
import numpy as np
import tensorflow as T
from PIL import Image
import string
import random
import warnings

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')

# --- 1. Define Constants & Helpers ---
# These MUST match the training script perfectly
IMAGE_WIDTH = 160
IMAGE_HEIGHT = 60
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits
NUM_CHARACTERS = len(CHARACTERS)
DATASET_DIR = "captcha_dataset_complex"
MODEL_FILE = "captcha_solver_alnum.keras"

# We only need int_to_char for decoding
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

# --- 2. Load the Trained Model ---
# We define a simple function to load the model so it doesn't
# reload every time if it's already in memory.
def load_model():
    """Loads the compiled Keras model from file."""
    if not os.path.exists(MODEL_FILE):
        print(f"ERROR: Model file not found: {MODEL_FILE}")
        print("Please make sure you have run the training script first.")
        return None
    try:
        model = T.keras.models.load_model(MODEL_FILE)
        print("Model loaded successfully.")
        return model
    except Exception as e:
        print(f"An error occurred while loading the model: {e}")
        return None

# --- 3. Pre-processing Function for a Single Image ---
def preprocess_for_prediction(image_path):
    """Loads and prepares a single image for prediction."""
    try:
        # Open as RGB (to match 3-channel input)
        image = Image.open(image_path).convert('RGB')

        # Check if size matches
        if image.size != (IMAGE_WIDTH, IMAGE_HEIGHT):
             print(f"Warning: Image {image_path} is {image.size}, resizing to {(IMAGE_WIDTH, IMAGE_HEIGHT)}.")
             image = image.resize((IMAGE_WIDTH, IMAGE_HEIGHT))

        image = np.array(image)
        image = image / 255.0
        # Reshape to (1, H, W, 3) for a single prediction
        image = image.reshape((1, IMAGE_HEIGHT, IMAGE_WIDTH, 3))
        return image
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

# --- 4. Run Prediction on a Random Image ---
def predict_random_image(model):
    """Picks a random image, preprocesses it, and predicts."""
    if model is None:
        print("Model is not loaded. Cannot run prediction.")
        return

    print("\n--- Testing with a new random sample ---")

    try:
        all_images = [f for f in os.listdir(DATASET_DIR) if f.endswith(".png")]
        if not all_images:
            print(f"ERROR: No images found in '{DATASET_DIR}' directory.")
            return

        # Pick one at random
        random_image_file = random.choice(all_images)
        image_path = os.path.join(DATASET_DIR, random_image_file)

        true_text = random_image_file.split('.')[0]

        sample_image = preprocess_for_prediction(image_path)

        if sample_image is not None:
            predictions = model.predict(sample_image, verbose=0)

            pred_labels_int = [np.argmax(char_prob) for char_prob in predictions[0]]
            pred_text = "".join([int_to_char[i] for i in pred_labels_int])

            print(f"\nImage File:     {random_image_file}")
            print(f"True Label:     {true_text}")
            print(f"Predicted Label:  {pred_text}")

            if true_text == pred_text:
                print("Result: SUCCESS!")
            else:
                print("Result: FAILED")
    except FileNotFoundError:
        print(f"ERROR: Dataset directory not found: {DATASET_DIR}")
    except Exception as e:
        print(f"An error occurred during prediction: {e}")


# --- 5. Main Execution (for Colab) ---
# Load the model once
# 'model' will be stored in the cell's global state
if 'model' not in locals() or 'model' is None:
    model = load_model()

# Now you can call this function repeatedly in the cell
if 'model' in locals() and model is not None:
    predict_random_image(model)

In [ ]:
#Evaluation Metrics
import os
import numpy as np
import tensorflow as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import string
import warnings

# Suppress TensorFlow/UserWarnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')

# --- Configuration (MUST MATCH YOUR TRAINING SCRIPT) ---
DATASET_DIR = "captcha_dataset_complex"
MODEL_FILE = "captcha_solver_alnum.keras"
IMAGE_WIDTH = 160
IMAGE_HEIGHT = 60
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits
NUM_CHARACTERS = len(CHARACTERS)
BATCH_SIZE = 64

# --- 1. Load Model and Data ---

print(f"Loading model {MODEL_FILE}...")
model = T.keras.models.load_model(MODEL_FILE)
print("Model loaded.")

char_to_int = {char: i for i, char in enumerate(CHARACTERS)}
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

def load_paths_and_labels():
    """Loads all file paths and labels from the dataset."""
    print("Loading all file paths and labels...")
    filenames = []
    labels = []
    for filename in os.listdir(DATASET_DIR):
        if filename.endswith(".png"):
            label_text = filename.split('.')[0]
            if len(label_text) != CAPTCHA_LENGTH:
                continue
            try:
                label_int = [char_to_int[char] for char in label_text]
                label_ohe = [to_categorical(i, num_classes=NUM_CHARACTERS) for i in label_int]
                label_stacked = np.stack(label_ohe)
                filenames.append(os.path.join(DATASET_DIR, filename))
                labels.append(label_stacked)
            except KeyError:
                continue
    return filenames, np.array(labels)

@T.function
def tf_load_image(path, label):
    """Loads one image (for evaluation, no augmentation)."""
    image_bytes = T.io.read_file(path)
    image = T.io.decode_png(image_bytes, channels=3)
    image = T.image.convert_image_dtype(image, T.float32)
    image = T.reshape(image, [IMAGE_HEIGHT, IMAGE_WIDTH, 3])
    return image, label

def create_dataset(filenames, labels):
    """Creates a non-shuffled tf.data pipeline for evaluation."""
    dataset = T.data.Dataset.from_tensor_slices((filenames, labels))
    dataset = dataset.map(tf_load_image, num_parallel_calls=T.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=T.data.AUTOTUNE)
    return dataset

# --- 2. Plot Training History ---
def plot_history(history):
    """Plots the model's training and validation accuracy/loss."""
    print("\nPlotting training history...")
    try:
        acc = history.history['accuracy']
        val_acc = history.history['val_accuracy']
        loss = history.history['loss']
        val_loss = history.history['val_loss']
        epochs = range(1, len(acc) + 1)

        plt.figure(figsize=(14, 5))

        plt.subplot(1, 2, 1)
        plt.plot(epochs, acc, 'bo-', label='Training Accuracy')
        plt.plot(epochs, val_acc, 'r*-', label='Validation Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(epochs, loss, 'bo-', label='Training Loss')
        plt.plot(epochs, val_loss, 'r*-', label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        plt.tight_layout()
        plt.show()
    except KeyError:
        print("Could not find 'accuracy' or 'loss' in history. Skipping plot.")
    except NameError:
        print("\nNOTE: 'history' object not found.")
        print("Please make sure you run this in the same session after training,")
        print("or re-run the training cell to generate the 'history' object.")


# --- 3. "Hardcoded" Advanced Metrics (F1, TPR, FPR) ---

def calculate_hardcoded_metrics(y_true_ohe, y_pred_prob, num_classes, characters):
    """
    Calculates per-position, macro-averaged metrics from raw predictions.
    This is complex because it's multi-output AND multi-class.
    """
    print("\nCalculating 'hardcoded' metrics (This is a complex task)...")

    # y_true_ohe shape is (num_samples, 5, 36)
    # y_pred_prob shape is (num_samples, 5, 36)

    # Convert from probabilities/OHE to class indices
    y_true_idx = np.argmax(y_true_ohe, axis=-1) # Shape: (num_samples, 5)
    y_pred_idx = np.argmax(y_pred_prob, axis=-1) # Shape: (num_samples, 5)

    num_samples = y_true_idx.shape[0]

    # We will calculate metrics for each of the 5 positions
    for pos in range(CAPTCHA_LENGTH):
        print(f"\n--- Metrics for Character Position {pos+1} ---")

        # Arrays to store TP, FP, FN for each of the 36 classes
        tp = np.zeros(num_classes, dtype=int)
        fp = np.zeros(num_classes, dtype=int)
        fn = np.zeros(num_classes, dtype=int)

        # Iterate over every sample's prediction for this position
        for i in range(num_samples):
            true_label = y_true_idx[i, pos]
            pred_label = y_pred_idx[i, pos]

            if true_label == pred_label:
                tp[true_label] += 1 # Correctly predicted this class
            else:
                fp[pred_label] += 1 # Falsely predicted this class
                fn[true_label] += 1 # Falsely predicted something else

        # Now calculate Precision, Recall, F1 for each class
        # We add a tiny epsilon (1e-6) to avoid division by zero
        epsilon = 1e-6
        precision = tp / (tp + fp + epsilon)
        recall = tp / (tp + fn + epsilon) # Recall is the same as TPR
        f1_score = 2 * (precision * recall) / (precision + recall + epsilon)

        # We can also calculate FPR (False Positive Rate) per class
        # TN = All samples - TP - FP - FN for that class
        tn = num_samples - tp - fp - fn
        fpr = fp / (fp + tn + epsilon)

        # Now we "macro-average" them (simple mean) to get one number
        macro_avg_precision = np.mean(precision)
        macro_avg_recall_tpr = np.mean(recall)
        macro_avg_f1 = np.mean(f1_score)
        macro_avg_fpr = np.mean(fpr)

        print(f"  Macro-Averaged Precision: {macro_avg_precision*100:.2f}%")
        print(f"  Macro-Averaged Recall (TPR): {macro_avg_recall_tpr*100:.2f}%")
        print(f"  Macro-Averaged F1-Score: {macro_avg_f1*100:.2f}%")
        print(f"  Macro-Averaged FPR: {macro_avg_fpr*100:.2f}%")

        # We can also plot a confusion matrix for this position
        pos_true_labels = y_true_idx[:, pos]
        pos_pred_labels = y_pred_idx[:, pos]

        cm = confusion_matrix(pos_true_labels, pos_pred_labels, labels=range(num_classes))

        plt.figure(figsize=(14, 10))
        sns.heatmap(cm, annot=False, fmt='d',
                    xticklabels=list(characters), yticklabels=list(characters))
        plt.title(f"Confusion Matrix for Position {pos+1}")
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()

# --- 4. ROC Curve Explanation ---
def explain_roc_curves():
    print("\n--- A Note on ROC Curves ---")
    print("A standard ROC (Receiver Operating Characteristic) curve is a tool for BINARY (2-class) classification.")
    print("Our model is a 'multi-output, multi-class' problem (5 outputs, 36 classes each).")
    print("We can't plot one single ROC curve for the whole model.")
    print("Instead, we would have to plot 5 * 36 = 180 different 'One-vs-All' ROC curves (e.g., 'A' vs. 'Not A' for position 1, 'B' vs. 'Not B' for position 1, etc.).")
    print("This is not very practical. A Confusion Matrix (plotted above) and the F1-Score are much better metrics for this type of problem, as they show exactly which characters are being confused.")


# --- Main execution ---
def main_evaluation():
    # 1. Load all data (this is memory-intensive, but needed for metrics)
    filenames, all_labels = load_paths_and_labels()

    # We only need the TEST set for a fair evaluation
    # We split with the *same* random_state to get the *same* test set
    _, X_test_paths, _, y_test = train_test_split(
        filenames, all_labels, test_size=TEST_SPLIT_SIZE, random_state=42
    )

    print(f"\nFound {len(X_test_paths)} test samples.")

    # 2. Create the test data pipeline
    test_dataset = create_dataset(X_test_paths, y_test)

    # 3. Get all predictions for the test set
    print("Running predictions on the entire test set (this may take a minute)...")
    y_pred = model.predict(test_dataset)
    print("Predictions complete.")

    # 4. Plot the training history
    # IMPORTANT: This line requires the 'history' object from your training cell
    # If it's not in memory, this function will just print a note.
    if 'history' in locals() or 'history' in globals():
        plot_history(history)
    else:
        print("\n'history' object not found. Run this in the same session as training to see history plots.")


    # 5. Calculate and show hardcoded metrics + confusion matrices
    # y_test is (20000, 5, 36)
    # y_pred is (20000, 5, 36)
    calculate_hardcoded_metrics(y_test, y_pred, NUM_CHARACTERS, CHARACTERS)

    # 6. Explain ROC
    explain_roc_curves()

if __name__ == "__main__":
    main_evaluation()

# In Colab, just run this cell.
# Make sure 'history' is in memory from the previous cell if you want the first plot.
main_evaluation()

In [ ]:
#Explainable ai
import os
import numpy as np
import tensorflow as T
from PIL import Image
import string
import random
import matplotlib.pyplot as plt
import cv2 # We need OpenCV for heatmap processing
import warnings

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')

# --- 1. Configuration (MUST MATCH YOUR TRAINING SCRIPT) ---
DATASET_DIR = "captcha_dataset_complex"
MODEL_FILE = "captcha_solver_alnum.keras"
IMAGE_WIDTH = 160
IMAGE_HEIGHT = 60
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits
NUM_CHARACTERS = len(CHARACTERS)

# We need the name of the *last convolutional layer* in our model
# From the summary, we know it's 'conv2d_32' (or similar, check your summary)
# Let's find it automatically
LAST_CONV_LAYER_NAME = None

# We also need the name of the *final output* before the Reshape/Softmax
# From our summary, it was 'unified_output_dense'
CLASSIFIER_ACTIVATION_LAYER = "unified_output_dense"


# --- 2. Dictionaries & Model Loading ---
char_to_int = {char: i for i, char in enumerate(CHARACTERS)}
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

print(f"Loading model {MODEL_FILE}...")
try:
    model = T.keras.models.load_model(MODEL_FILE)
    # Automatically find the last Conv2D layer
    for layer in reversed(model.layers):
        if 'conv2d' in layer.name:
            LAST_CONV_LAYER_NAME = layer.name
            break
    print(f"Model loaded. Using last conv layer: '{LAST_CONV_LAYER_NAME}' for Grad-CAM.")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# --- 3. Grad-CAM Implementation ---
def make_grad_cam_heatmap(image_array, model, last_conv_layer_name, classifier_activation_layer):
    """
    Generates a Grad-CAM heatmap.
    This function creates a new "explanation model" that outputs
    both the final prediction and the feature maps of the last conv layer.
    """

    # Create a model that maps the input image to the activations
    # of the last conv layer as well as the final output
    grad_model = T.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.get_layer(classifier_activation_layer).output]
    )

    # Use GradientTape to get the gradients of the top predicted class
    # with respect to the output of the last conv layer
    with T.GradientTape() as tape:
        # Get the conv layer outputs and the final predictions
        conv_outputs, predictions = grad_model(image_array)

        # We need to find the "top" prediction.
        # But our output is (1, 180). We'll focus on the *single most active neuron*
        # in the entire (1, 180) output vector.
        top_pred_index = T.argmax(predictions[0])
        top_class_channel = predictions[:, top_pred_index]

    # This is the gradient of the top predicted class with regard to the
    # output feature map of the last conv layer
    grads = tape.gradient(top_class_channel, conv_outputs)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a specific feature map channel
    pooled_grads = T.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array
    # by "how important this channel is" (the pooled gradient)
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., T.newaxis]
    heatmap = T.squeeze(heatmap)

    # To make it visual, we'll normalize the heatmap
    heatmap = T.maximum(heatmap, 0) / T.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(image_array, heatmap, alpha=0.6, colormap=cv2.COLORMAP_JET):
    """
    Applies the heatmap as an overlay on the original image.
    """
    # Resize heatmap to match original image size
    heatmap = cv2.resize(heatmap, (image_array.shape[1], image_array.shape[0]))

    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, colormap)

    # Convert original image to RGB (if it's grayscale)
    if image_array.shape[2] == 1:
        image_array = cv2.cvtColor(image_array, cv2.COLOR_GRAY2RGB)

    image_array = np.uint8(255 * image_array)

    # Superimpose the heatmap
    superimposed_img = cv2.addWeighted(image_array, 1 - alpha, heatmap, alpha, 0)

    return superimposed_img

# --- 4. Pre-processing and Prediction Function ---
def preprocess_for_prediction(image_path):
    """Loads and prepares a single image for prediction."""
    image = Image.open(image_path).convert('RGB')
    if image.size != (IMAGE_WIDTH, IMAGE_HEIGHT):
         image = image.resize((IMAGE_WIDTH, IMAGE_HEIGHT))
    image_array = np.array(image) / 255.0
    # Reshape to (1, H, W, 3) for the model
    image_for_model = image_array.reshape((1, IMAGE_HEIGHT, IMAGE_WIDTH, 3))
    # Return the original array for plotting and the model-ready array
    return image_array, image_for_model

# --- 5. Main "Explain" Function ---
def explain_random_image():
    if model is None or LAST_CONV_LAYER_NAME is None:
        print("Model not loaded. Cannot run explanation.")
        return

    print("\n--- Explaining a Random Prediction (Grad-CAM) ---")

    try:
        all_images = [f for f in os.listdir(DATASET_DIR) if f.endswith(".png")]
        if not all_images:
            print(f"ERROR: No images found in '{DATASET_DIR}'.")
            return

        random_image_file = random.choice(all_images)
        image_path = os.path.join(DATASET_DIR, random_image_file)

        # Get true label and preprocess image
        true_text = random_image_file.split('.')[0]
        original_image_array, image_for_model = preprocess_for_prediction(image_path)

        # Make prediction
        predictions = model.predict(image_for_model, verbose=0)
        pred_labels_int = [np.argmax(char_prob) for char_prob in predictions[0]]
        pred_text = "".join([int_to_char[i] for i in pred_labels_int])

        print(f"Image File:     {random_image_file}")
        print(f"True Label:     {true_text}")
        print(f"Predicted Label:  {pred_text}")

        # Generate the Grad-CAM heatmap
        heatmap = make_grad_cam_heatmap(
            image_for_model,
            model,
            LAST_CONV_LAYER_NAME,
            CLASSIFIER_ACTIVATION_LAYER
        )

        # Overlay heatmap on original image
        superimposed_image = overlay_heatmap(original_image_array, heatmap)

        # Plot
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(original_image_array)
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(superimposed_image)
        plt.title("Grad-CAM: Why the model predicted this")
        plt.axis('off')

        plt.show()

    except Exception as e:
        print(f"An error occurred during explanation: {e}")

# --- 6. Execute ---
# In Colab, you can just call this function in the cell to test:
# explain_random_image()

explain_random_image()

Best Model (No Changes !!!!!!!!!)

In [ ]:
import os
import random
import string
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# --- Configuration ---
# This is the ORIGINAL noisy generator. This is what we want to solve.
CAPTCHA_LENGTH = 5
IMAGE_WIDTH = 200
IMAGE_HEIGHT = 70
CHARACTERS = string.ascii_uppercase + string.digits
DATASET_DIR = "captcha_dataset_final"
NUM_IMAGES_TO_GENERATE = 10# 5000 is a good start, 10k is better

def get_font():
    try:
        font = ImageFont.truetype("arial.ttf", 400)
    except IOError:
        try:
            font = ImageFont.truetype("DejaVuSans.ttf", 400)
        except IOError:
            print("Font not found. Using default PIL font.")
            font = ImageFont.load_default()
    return font

def generate_random_text(length):
    """Generates a random string of the specified length."""
    return ''.join(random.choices(CHARACTERS, k=length))

def generate_captcha_image(font):
    """
    Generates a single NOISY CAPTCHA image.
    This is our real, hard problem.
    """
    image = Image.new('L', (IMAGE_WIDTH, IMAGE_HEIGHT), color='white')
    draw = ImageDraw.Draw(image)

    text = generate_random_text(CAPTCHA_LENGTH)

    text_width, text_height = draw.textbbox((0, 0), text, font=font)[2:]
    x = (IMAGE_WIDTH - text_width) / 2
    y = (IMAGE_HEIGHT - text_height) / 2

    current_x = x
    for char in text:
        char_width, char_height = draw.textbbox((0, 0), char, font=font)[2:]
        char_y = y + random.randint(-5, 5)
        char_x = current_x + random.randint(-3, 3)
        draw.text((char_x, char_y), char, fill='black', font=font)
        current_x += char_width + random.randint(-2, 2)

    # Add random dots (salt-and-pepper noise)
    for _ in range(random.randint(200, 400)):
        x_dot = random.randint(0, IMAGE_WIDTH - 1)
        y_dot = random.randint(0, IMAGE_HEIGHT - 1)
        draw.point((x_dot, y_dot), fill='black')

    # Add random lines
    for _ in range(random.randint(3, 7)):
        x1 = random.randint(0, IMAGE_WIDTH)
        y1 = random.randint(0, IMAGE_HEIGHT)
        x2 = random.randint(0, IMAGE_WIDTH)
        y2 = random.randint(0, IMAGE_HEIGHT)
        draw.line((x1, y1, x2, y2), fill='black', width=random.randint(1, 2))

    # Apply distortions
    image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.8)))
    image = image.filter(ImageFilter.SMOOTH)

    return image, text

def create_dataset():
    """Generates and saves the CAPTCHA images to the dataset directory."""
    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATASET_DIR)
        print(f"Created directory: {DATASET_DIR}")
    else:
        print(f"Directory {DATASET_DIR} found. Deleting old images...")
        for f in os.listdir(DATASET_DIR):
            os.remove(os.path.join(DATASET_DIR, f))
        print("Old images deleted.")

    font = get_font()

    print(f"Generating {NUM_IMAGES_TO_GENERATE} NOISY CAPTCHA images...")
    for i in range(NUM_IMAGES_TO_GENERATE):
        image, text = generate_captcha_image(font)
        filename = os.path.join(DATASET_DIR, f"{text}.png")

        while os.path.exists(filename):
            image, text = generate_captcha_image(font)
            filename = os.path.join(DATASET_DIR, f"{text}.png")

        image.save(filename)

        if (i + 1) % 100 == 0:
            print(f"  ... generated {i + 1} / {NUM_IMAGES_TO_GENERATE}")

    print("New dataset generation complete.")

if __name__ == "__main__":
    create_dataset()

Directory captcha_dataset_final found. Deleting old images...
Old images deleted.
Font not found. Using default PIL font.
Generating 10 NOISY CAPTCHA images...
New dataset generation complete.


In [ ]:
import os
import numpy as np
import tensorflow as T
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Dense, Dropout,
    Flatten, BatchNormalization, Reshape, Softmax,
    Bidirectional, LSTM
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import string

# --- Configuration ---
DATASET_DIR = "captcha_dataset"
IMAGE_WIDTH = 200
IMAGE_HEIGHT = 70
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits
NUM_CHARACTERS = len(CHARACTERS)
TEST_SPLIT_SIZE = 0.2
BATCH_SIZE = 64
EPOCHS = 100 # EarlyStopping will find the best epoch

# --- 1. Data Loading (NEW tf.data pipeline) ---

char_to_int = {char: i for i, char in enumerate(CHARACTERS)}
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

def load_paths_and_labels():
    """
    Loads just the file paths and labels, not the images.
    This is much faster and saves memory.
    """
    print("Loading file paths and labels...")
    filenames = []
    labels = []

    for filename in os.listdir(DATASET_DIR):
        if filename.endswith(".png"):
            label_text = filename.split('.')[0]

            if len(label_text) != CAPTCHA_LENGTH:
                print(f"Skipping file with incorrect length: {filename}")
                continue

            try:
                # 2. Label Encoding (Unified Shape)
                label_int = [char_to_int[char] for char in label_text]
                label_ohe = [to_categorical(i, num_classes=NUM_CHARACTERS) for i in label_int]
                # Stack into a single (5, 36) array
                label_stacked = np.stack(label_ohe)

                filenames.append(os.path.join(DATASET_DIR, filename))
                labels.append(label_stacked)
            except KeyError:
                print(f"Skipping file with unknown character: {filename}")

    return filenames, labels

@T.function
def tf_load_and_augment_image(path, label):
    """
    Loads one image, decodes it, and applies data augmentation.
    This runs in the TensorFlow graph for high performance.
    """
    # 1. Read the file from disk
    image_bytes = T.io.read_file(path)
    # 2. Decode as grayscale
    image = T.io.decode_png(image_bytes, channels=1)
    # 3. Convert to float32 and normalize
    image = T.image.convert_image_dtype(image, T.float32)
    image = T.reshape(image, [IMAGE_HEIGHT, IMAGE_WIDTH, 1])

    # --- 4. Data Augmentation ---
    # This is the KEY to preventing overfitting.
    # The model never sees the *exact* same image twice.
    image = T.image.random_brightness(image, max_delta=0.2)
    image = T.image.random_contrast(image, lower=0.8, upper=1.2)
    # Clip values to ensure they stay between 0 and 1
    image = T.clip_by_value(image, 0.0, 1.0)

    return image, label

def create_dataset(filenames, labels, is_training=True):
    """
    Creates a high-performance tf.data.Dataset pipeline.
    """
    dataset = T.data.Dataset.from_tensor_slices((filenames, labels))

    if is_training:
        # For training, we shuffle the data
        dataset = dataset.shuffle(buffer_size=len(filenames))

    # Apply the loading and augmentation
    # num_parallel_calls=T.data.AUTOTUNE lets TF handle the parallelism
    dataset = dataset.map(tf_load_and_augment_image, num_parallel_calls=T.data.AUTOTUNE)

    # Batch the data
    dataset = dataset.batch(BATCH_SIZE)

    # Prefetch the next batch in the background for speed
    dataset = dataset.prefetch(buffer_size=T.data.AUTOTUNE)

    return dataset


# --- 2. Build the FINAL CRNN Model (Same as before) ---

def build_crnn_model():
    """
    Builds the definitive CRNN (Convolutional Recurrent Neural Network)
    architecture.
    """
    print("Building final CRNN model...")

    # Input Layer
    input_layer = Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 1), name="image_input")

    # --- 1. The "Eye" (CNN Base) ---
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    # CNN Output shape: (None, 4, 12, 256)

    # --- 2. The "Bridge" (Reshape) ---
    cnn_output_shape = x.shape[1:]
    new_shape = (cnn_output_shape[1], cnn_output_shape[0] * cnn_output_shape[2])
    x = Reshape(target_shape=new_shape, name="reshape_to_sequence")(x)

    # --- 3. The "Reader" (RNN) ---
    x = Bidirectional(LSTM(128, return_sequences=True), name="bidirectional_lstm")(x)

    # --- 4. The "Hand" (Unified Predictor) ---
    x = Flatten(name="flatten_smart_sequence")(x)
    x = Dense(512, activation='relu', name="dense_features")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    output_neurons = CAPTCHA_LENGTH * NUM_CHARACTERS # 5 * 36 = 180
    x = Dense(output_neurons, name="unified_output_dense")(x)
    x = Reshape((CAPTCHA_LENGTH, NUM_CHARACTERS), name="output_reshape")(x)
    output_layer = Softmax(axis=-1, name="output_softmax")(x)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    model.summary()
    return model

# --- 3. Train and Test ---

def main():
    # 1. Load paths and labels
    filenames, labels = load_paths_and_labels()

    # 2. Split into training and testing sets
    X_train_paths, X_test_paths, y_train, y_test = train_test_split(
        filenames, labels, test_size=TEST_SPLIT_SIZE, random_state=42
    )

    print(f"Training data: {len(X_train_paths)} samples")
    print(f"Testing data: {len(X_test_paths)} samples")

    # 3. Create the tf.data.Dataset pipelines
    print("Creating data pipelines...")
    train_dataset = create_dataset(X_train_paths, y_train, is_training=True)
    val_dataset = create_dataset(X_test_paths, y_test, is_training=False)

    # 4. Build the model
    model = build_crnn_model()

    # 5. Define callbacks
    print("Defining callbacks (EarlyStopping, ReduceLROnPlateau)...")

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10, # Stop if val_loss doesn't improve for 10 epochs
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2, # new_lr = lr * 0.2
        patience=5,
        min_lr=0.00001
    )

    # 6. Train the model
    print("Training final CRNN model with augmentation...")
    history = model.fit(
        train_dataset,  # <-- Pass the pipeline directly
        validation_data=val_dataset, # <-- Pass the pipeline directly
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr]
        # No batch_size here, it's defined in the pipeline
    )

    # 7. Evaluate the model
    print("\nEvaluating model on test set...")
    metrics = model.evaluate(val_dataset, verbose=0)

    print("\n--- Evaluation ---")
    print(f"Total Loss: {metrics[0]:.4f}")
    print(f"  Overall Character Accuracy: {metrics[1]*100:.2f}%")

    # 8. Save the model
    model.save("captcha_solver_CRNN_AUGMENTED.keras")
    print("\nModel saved to 'captcha_solver_CRNN_AUGMENTED.keras'")

    # 9. Test on a random sample from the test set
    print("\n--- Testing with a random sample ---")

    # Get a single batch from the test set
    test_batch_images, test_batch_labels = next(iter(val_dataset))

    # Pick one image from the batch
    idx = np.random.randint(0, BATCH_SIZE)
    sample_image = np.expand_dims(test_batch_images[idx], axis=0) # (1, H, W, 1)
    sample_label = test_batch_labels[idx] # (5, 36)

    true_labels_int = [np.argmax(char_ohe) for char_ohe in sample_label]
    true_text = "".join([int_to_char[i] for i in true_labels_int])

    # Make a prediction
    predictions = model.predict(sample_image)

    pred_labels_int = [np.argmax(char_prob) for char_prob in predictions[0]]
    pred_text = "".join([int_to_char[i] for i in pred_labels_int])

    print(f"True Label:     {true_text}")
    print(f"Predicted Label:  {pred_text}")

    if true_text == pred_text:
        print("Result: SUCCESS!")
    else:
        print("Result: FAILED")

if __name__ == "__main__":
    main()

Loading file paths and labels...
Training data: 80000 samples
Testing data: 20000 samples
Creating data pipelines...
Building final CRNN model...


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 70, 200, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 70, 200, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 70, 200, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 35, 100, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 35, 100, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 35, 100, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 17, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 17, 50, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 17, 50, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_31 (MaxPooling2D) │ (None, 8, 25, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 8, 25, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 8, 25, 256)     │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_32 (MaxPooling2D) │ (None, 4, 12, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_to_sequence (Reshape)   │ (None, 12, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_lstm              │ (None, 12, 256)        │     1,180,672 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_smart_sequence          │ (None, 3072)           │             0 │
│ (Flatten)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_features (Dense)          │ (None, 512)            │     1,573,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ unified_output_dense (Dense)    │ (None, 180)            │        92,340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_reshape (Reshape)        │ (None, 5, 36)          │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 3,238,196 (12.35 MB)

 Trainable params: 3,236,212 (12.35 MB)

 Non-trainable params: 1,984 (7.75 KB)

Defining callbacks (EarlyStopping, ReduceLROnPlateau)...
Training final CRNN model with augmentation...
Epoch 1/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 126s 97ms/step - accuracy: 0.0303 - loss: 3.9073 - val_accuracy: 0.0431 - val_loss: 3.6701 - learning_rate: 0.0010
Epoch 2/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 136s 92ms/step - accuracy: 0.1652 - loss: 2.9514 - val_accuracy: 0.3670 - val_loss: 2.1427 - learning_rate: 0.0010
Epoch 3/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 121s 97ms/step - accuracy: 0.4182 - loss: 1.9687 - val_accuracy: 0.4070 - val_loss: 2.0355 - learning_rate: 0.0010
Epoch 4/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 108s 86ms/step - accuracy: 0.5326 - loss: 1.5870 - val_accuracy: 0.4699 - val_loss: 1.8753 - learning_rate: 0.0010
Epoch 5/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 141s 85ms/step - accuracy: 0.5930 - loss: 1.3808 - val_accuracy: 0.5967 - val_loss: 1.4096 - learning_rate: 0.0010
Epoch 6/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 108s 86ms/step - accuracy: 0.6331 - loss: 1.2433 - val_accuracy

In [ ]:
#random predict
import os
import numpy as np
import tensorflow as T
from PIL import Image
import string
import random

# --- 1. Define Constants & Helpers ---
# These must be the SAME as what you trained with
IMAGE_WIDTH = 200
IMAGE_HEIGHT = 70
CAPTCHA_LENGTH = 5
CHARACTERS = string.ascii_uppercase + string.digits
NUM_CHARACTERS = len(CHARACTERS)
DATASET_DIR = "captcha_dataset"

# We only need int_to_char for decoding
int_to_char = {i: char for i, char in enumerate(CHARACTERS)}

# --- 2. Load the Trained Model ---
MODEL_FILE = "captcha_solver_CRNN_AUGMENTED.keras"

if not os.path.exists(MODEL_FILE):
    print(f"ERROR: Model file not found: {MODEL_FILE}")
    print("Please make sure the file is in the same directory.")
else:
    print(f"Loading saved model from {MODEL_FILE}...")
    try:
        # Load the compiled model
        model = T.keras.models.load_model(MODEL_FILE)
        print("Model loaded successfully.")
    except Exception as e:
        print(f"An error occurred while loading the model: {e}")
        model = None

    # --- 3. Pre-processing Function for a Single Image ---
    def preprocess_for_prediction(image_path):
        """Loads and prepares a single image for prediction."""
        try:
            image = Image.open(image_path).convert('L')
            image = np.array(image)
            image = image / 255.0
            # Reshape to (1, H, W, 1) for a single prediction
            image = image.reshape((1, IMAGE_HEIGHT, IMAGE_WIDTH, 1))
            return image
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")
            return None

    # --- 4. Run Prediction on a Random Image ---
    if model:
        print("\n--- Testing with a new random sample ---")

        # Get a list of all images in the dataset
        try:
            all_images = [f for f in os.listdir(DATASET_DIR) if f.endswith(".png")]
            if not all_images:
                print(f"ERROR: No images found in '{DATASET_DIR}' directory.")
            else:
                # Pick one at random
                random_image_file = random.choice(all_images)
                image_path = os.path.join(DATASET_DIR, random_image_file)

                # Get the true label from the filename
                true_text = random_image_file.split('.')[0]

                # Pre-process the image
                sample_image = preprocess_for_prediction(image_path)

                if sample_image is not None:
                    # Make a prediction
                    predictions = model.predict(sample_image)

                    # Decode the prediction
                    # predictions[0] is shape (5, 36)
                    pred_labels_int = [np.argmax(char_prob) for char_prob in predictions[0]]
                    pred_text = "".join([int_to_char[i] for i in pred_labels_int])

                    print(f"\nImage File:     {random_image_file}")
                    print(f"True Label:     {true_text}")
                    print(f"Predicted Label:  {pred_text}")

                    if true_text == pred_text:
                        print("Result: SUCCESS!")
                    else:
                        print("Result: FAILED")
        except FileNotFoundError:
            print(f"ERROR: Dataset directory not found: {DATASET_DIR}")
        except Exception as e:
            print(f"An error occurred during prediction: {e}")

Loading saved model from captcha_solver_CRNN_AUGMENTED.keras...
Model loaded successfully.

--- Testing with a new random sample ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 554ms/step

Image File:     15IOW.png
True Label:     15IOW
Predicted Label:  1IIOW
Result: FAILED
